# Plant Species Classification — Sklearn + EfficientNet-B0

Two-stage pipeline:
1. **Feature extraction** — frozen EfficientNet-B0 (pretrained on ImageNet) produces a 1280-dim embedding per image. No training happens here.
2. **Sklearn classifiers** — SVM, LogisticRegression, and RandomForest are trained on those embeddings and compared.

## Running locally vs. Google Colab

| Setting | Action |
|---|---|
| **Local** | Set `COLAB = False` in Cell 2. Paths point to your local repo. |
| **Google Colab (GPU)** | Set `COLAB = True`. Mount your Google Drive and update `PROJECT_ROOT` to where you uploaded the dataset. Feature extraction is ~30-60s on a T4 GPU vs. ~3-4 min on CPU. |

In [ ]:
# ── Colab GPU setup — run this cell first, then restart the runtime ─────────
import subprocess, sys

_in_colab = "google.colab" in sys.modules or "COLAB_BACKEND_VERSION" in __import__("os").environ

if _in_colab:
    subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"])

    print("Installing dependencies — restart the runtime when done...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "torch", "torchvision",
        "--index-url", "https://download.pytorch.org/whl/cu121",
    ])
    # Pin sklearn to match the local environment — avoids InconsistentVersionWarning
    # when loading the pickled model locally. Update this if your local version changes.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.8.0"])

    print("Done. Please restart the runtime now (Runtime → Restart session).")
else:
    import sklearn
    print(f"Local environment — sklearn {sklearn.__version__}")

In [ ]:
from pathlib import Path

# ── Set this to True when running on Google Colab ──────────────────────────
COLAB = True

if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT   = Path("/content/drive/MyDrive/bootcamp_project")  # ← adjust to your Drive path
    ALL_IMAGES_DIR = PROJECT_ROOT / "data/raw/all_images"
    MODELS_DIR     = PROJECT_ROOT / "backend/app/models_sklearn"
else:
    PROJECT_ROOT   = Path("/home/jouell/code/jouell3/plant_detect")
    ALL_IMAGES_DIR = PROJECT_ROOT / "data/raw/all_images"
    MODELS_DIR     = PROJECT_ROOT / "backend/app/models_sklearn"

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"ALL_IMAGES_DIR exists: {ALL_IMAGES_DIR.exists()}")

In [ ]:
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
print("Imports OK")

In [ ]:
# Runtime configuration
BACKBONE     = "b3"   # "b0" (224px, 1280-dim) or "b3" (300px, 1536-dim)
RANDOM_STATE = 42
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Each EfficientNet variant was pretrained at a specific resolution — use it.
IMG_SIZE   = {"b0": 224, "b3": 300}[BACKBONE]
BATCH_SIZE = 128 if torch.cuda.is_available() else 64

print(f"Device     : {DEVICE}")
print(f"Backbone   : EfficientNet-{BACKBONE.upper()}  |  img size: {IMG_SIZE}px")
print(f"Batch size : {BATCH_SIZE}")

## 1 — Load labels

`data/files_df.csv` is the ground truth (columns: `filename`, `name`). Images not present in `all_images/` are silently dropped so the notebook stays robust as the dataset is updated.

In [ ]:
files_df = pd.read_csv(PROJECT_ROOT / "data/files_df.csv")

# Keep only rows whose image actually exists in all_images/
files_df = files_df[
    files_df["filename"].apply(lambda f: (ALL_IMAGES_DIR / f).exists())
].reset_index(drop=True)

print(files_df["name"].value_counts().to_string())
print(f"\nTotal images : {len(files_df)}")
print(f"Classes      : {files_df['name'].nunique()}")

In [ ]:
# ── Copy images from Drive to Colab local SSD (Colab only) ──────────────────
# Google Drive has ~100-200 ms latency per file read.
# Copying once to /content/ (local NVMe SSD) makes extraction ~100x faster.
import shutil
from concurrent.futures import ThreadPoolExecutor

if COLAB:
    LOCAL_IMAGES_DIR = Path("/content/plant_images")
    LOCAL_IMAGES_DIR.mkdir(exist_ok=True)

    filenames = files_df["filename"].tolist()
    already = {p.name for p in LOCAL_IMAGES_DIR.iterdir()}
    to_copy  = [fn for fn in filenames if fn not in already]

    if to_copy:
        print(f"Copying {len(to_copy)} images to local SSD...")
        def _copy(fn):
            shutil.copy2(ALL_IMAGES_DIR / fn, LOCAL_IMAGES_DIR / fn)
        with ThreadPoolExecutor(max_workers=8) as pool:
            list(tqdm(pool.map(_copy, to_copy), total=len(to_copy), desc="Copying"))
    else:
        print(f"All {len(filenames)} images already on local SSD.")

    # Point ALL_IMAGES_DIR to local disk for all subsequent cells
    ALL_IMAGES_DIR = LOCAL_IMAGES_DIR
    print(f"ALL_IMAGES_DIR → {ALL_IMAGES_DIR}")
else:
    print("Local run — no copy needed.")

## 2 — Encode labels and split dataset

Same stratified split strategy as the PyTorch notebook:
- **85 %** → train + validation
- **15 %** → held-out test set
- Inner split: 15 % of train+val → validation

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(files_df["name"].values)
NUM_CLASSES = len(le.classes_)
img_paths   = [str(ALL_IMAGES_DIR / fn) for fn in files_df["filename"]]

idx = np.arange(len(img_paths))

# Outer split: 85% train+val, 15% test
idx_trainval, idx_test = train_test_split(
    idx, test_size=0.15, stratify=y_enc, random_state=RANDOM_STATE
)
# Inner split: 15% of trainval → validation
idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=0.15,
    stratify=y_enc[idx_trainval],
    random_state=RANDOM_STATE,
)

print(f"Classes : {NUM_CLASSES}  →  {list(le.classes_)}")
print(f"Train   : {len(idx_train)}")
print(f"Val     : {len(idx_val)}")
print(f"Test    : {len(idx_test)}")

## 3 — Feature extractor backbone

A frozen EfficientNet pretrained on ImageNet is used as a fixed feature extractor. Two options are available:

| Backbone | Output dim | Model size | Expected accuracy |
|---|---|---|---|
| **EfficientNet-B0** (default) | 1280 | 20 MB | ~75% |
| **EfficientNet-B3** (recommended) | 1536 | 48 MB | ~79-81% |

Set `BACKBONE = "b3"` to get richer features at the cost of a slightly longer extraction (~90s on T4).

In [ ]:
# ── Choose backbone ──────────────────────────────────────────────────────────
# "b0"  →  EfficientNet-B0, 1280-dim features, faster extraction
# "b3"  →  EfficientNet-B3, 1536-dim features, better accuracy (~+4-6 pp)
BACKBONE = "b3"

if BACKBONE == "b3":
    backbone = models.efficientnet_b3(weights="IMAGENET1K_V1")
    FEAT_DIM = 1536
else:
    backbone = models.efficientnet_b0(weights="IMAGENET1K_V1")
    FEAT_DIM = 1280

backbone.classifier = nn.Identity()   # strip head → raw pooled features
backbone = backbone.to(DEVICE).eval()

# ImageNet normalisation (same for all EfficientNet variants)
preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

print(f"Backbone : EfficientNet-{BACKBONE.upper()}  |  output dim: {FEAT_DIM}")

## 4 — Extract features from all images

Features are re-extracted fresh every run so the matrix is always consistent with `files_df.csv` (important as the dataset grows).

**On Colab**: the previous cell copied all images to local SSD — extraction now takes ~30-60 s on a T4 GPU. Reading from Google Drive directly would take 30-40 min due to per-file network latency.

In [ ]:
def extract_features(paths: list, model, transform, device, batch_size: int) -> np.ndarray:
    """Run a frozen CNN over a list of image paths and return (N, D) feature matrix."""
    all_feats = []
    for start in tqdm(range(0, len(paths), batch_size), desc="Extracting features"):
        chunk = paths[start : start + batch_size]
        tensors = [transform(Image.open(p).convert("RGB")) for p in chunk]
        batch = torch.stack(tensors).to(device)
        with torch.no_grad():
            feats = model(batch)          # shape: (batch_size, 1280)
        all_feats.append(feats.cpu().numpy())
    return np.vstack(all_feats)


X_all = extract_features(img_paths, backbone, preprocess, DEVICE, BATCH_SIZE)
print(f"Feature matrix shape: {X_all.shape}")   # (N, 1280)

## 4b — K-Fold Cross-Validation Benchmarking

`StratifiedKFold` splits the *full* feature matrix `X_all` (already extracted above — no re-extraction per fold).
Each fold trains all four classifiers from scratch and records validation accuracy.
Mean ± std across folds shows how stable each classifier is regardless of the particular split.

The train/val/test split used to produce the saved model (sections 5-8) is **separate** from this benchmark.

In [ ]:
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)


def _make_classifiers():
    """Return a fresh set of classifier pipelines — called once per fold."""
    return {
        "SVM (RBF)": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=10.0, gamma="scale",
                class_weight="balanced", probability=False,
                random_state=RANDOM_STATE,
            )),
        ]),
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=1.0, max_iter=2000, solver="lbfgs",
                multi_class="multinomial", class_weight="balanced",
                random_state=RANDOM_STATE, n_jobs=-1,
            )),
        ]),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, max_depth=None,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "MLP": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(512, 256), activation="relu",
                alpha=1e-3, batch_size=256, max_iter=300,
                early_stopping=True, validation_fraction=0.1,
                n_iter_no_change=15, random_state=RANDOM_STATE, verbose=False,
            )),
        ]),
    }


kfold_scores = {name: [] for name in _make_classifiers()}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_enc)):
    print(f"\n── Fold {fold + 1}/{N_SPLITS} ──────────────────────────────────────")
    X_tr, X_vl = X_all[train_idx], X_all[val_idx]
    y_tr, y_vl = y_enc[train_idx], y_enc[val_idx]

    for name, clf in _make_classifiers().items():
        clf.fit(X_tr, y_tr)
        acc = clf.score(X_vl, y_vl)
        kfold_scores[name].append(acc)
        print(f"  {name:<22}  acc = {acc:.4f}")

# ── Summary table ─────────────────────────────────────────────────────────────
kfold_df = pd.DataFrame(
    kfold_scores,
    index=[f"Fold {i + 1}" for i in range(N_SPLITS)],
)
kfold_df.loc["Mean"] = kfold_df.mean()
kfold_df.loc["Std"]  = kfold_df.std()

print("\n── K-Fold Results ───────────────────────────────────────────────────────")
print(kfold_df.to_string(float_format="{:.4f}".format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: bar chart with error bars ──────────────────────────────────────────
clf_names = list(kfold_scores.keys())
means = [np.mean(kfold_scores[n]) for n in clf_names]
stds  = [np.std(kfold_scores[n])  for n in clf_names]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

bars = axes[0].bar(clf_names, means, yerr=stds, capsize=6,
                   color=colors, alpha=0.85,
                   error_kw=dict(elinewidth=1.5))
axes[0].set_ylabel("Accuracy")
axes[0].set_title(f"{N_SPLITS}-Fold CV — EfficientNet-{BACKBONE.upper()} embeddings")
axes[0].set_ylim(0, 1)
axes[0].bar_label(
    bars,
    labels=[f"{m:.3f}\n±{s:.3f}" for m, s in zip(means, stds)],
    padding=4, fontsize=9,
)
axes[0].tick_params(axis="x", rotation=15)

# ── Right: per-fold accuracy lines ───────────────────────────────────────────
fold_x = list(range(1, N_SPLITS + 1))
for name, color in zip(clf_names, colors):
    axes[1].plot(fold_x, kfold_scores[name], marker="o", label=name, color=color)

axes[1].set_xlabel("Fold")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Per-fold accuracy by classifier")
axes[1].set_xticks(fold_x)
axes[1].set_ylim(0, 1)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 5 — Apply split indices to feature matrix

In [ ]:
X_train, y_train = X_all[idx_train], y_enc[idx_train]
X_val,   y_val   = X_all[idx_val],   y_enc[idx_val]
X_test,  y_test  = X_all[idx_test],  y_enc[idx_test]

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}   y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}   y_test  : {y_test.shape}")

## 6 — Class weight computation

The dataset is imbalanced (some species have many more images than others). `compute_class_weight("balanced")` gives each class a weight inversely proportional to its frequency so all classifiers treat minority classes fairly.

In [ ]:
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train,
)
class_weight_dict = dict(enumerate(class_weights_array))

# Show 5 heaviest weights (minority classes)
top5 = sorted(class_weight_dict.items(), key=lambda x: x[1], reverse=True)[:5]
print("5 most under-represented classes:")
for idx_c, w in top5:
    print(f"  {le.classes_[idx_c]:<18}  weight = {w:.3f}")

## 7 — Train and compare classifiers

Four classifiers are trained on the EfficientNet embeddings and evaluated on the validation set:

| Classifier | Notes |
|---|---|
| **SVM (RBF kernel)** | Strong on high-dimensional embeddings. Slow to train at scale. |
| **Logistic Regression** | Fast, interpretable. Multinomial softmax handles multi-class well. |
| **Random Forest** | No scaling needed. Usually the weakest on continuous deep embeddings. |
| **MLP (neural network)** | sklearn's `MLPClassifier`. Two hidden layers learn non-linear feature combinations — typically the strongest of the four. Not a CNN, but the closest thing sklearn offers. |

> **Note**: sklearn has no CNN implementation. The backbone (EfficientNet) already extracts the convolutional features; the classifiers here operate purely on the resulting embedding vectors.

In [ ]:
results = {}

# ── 1. SVM with RBF kernel ───────────────────────────────────────────────────
print("Training SVM (RBF)...")
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="rbf",
        C=10.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE,
    )),
])
svm_pipe.fit(X_train, y_train)
val_acc_svm = svm_pipe.score(X_val, y_val)
results["SVM (RBF)"] = {"model": svm_pipe, "val_acc": val_acc_svm}
print(f"  Val accuracy: {val_acc_svm:.4f}\n")

# ── 2. Logistic Regression ───────────────────────────────────────────────────
print("Training Logistic Regression...")
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=2000,
        solver="lbfgs",
        multi_class="multinomial",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])
lr_pipe.fit(X_train, y_train)
val_acc_lr = lr_pipe.score(X_val, y_val)
results["Logistic Regression"] = {"model": lr_pipe, "val_acc": val_acc_lr}
print(f"  Val accuracy: {val_acc_lr:.4f}\n")

# ── 3. Random Forest ─────────────────────────────────────────────────────────
print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
val_acc_rf = rf.score(X_val, y_val)
results["Random Forest"] = {"model": rf, "val_acc": val_acc_rf}
print(f"  Val accuracy: {val_acc_rf:.4f}\n")

# ── 4. MLP (sklearn neural network) ─────────────────────────────────────────
# Two hidden layers (512 → 256) learn non-linear combinations of the embeddings.
# StandardScaler is required — MLP training is sensitive to feature scale.
# early_stopping uses 10% of training data to stop when val loss plateaus.
print("Training MLP...")
mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(512, 256),
        activation="relu",
        alpha=1e-3,           # L2 regularisation
        batch_size=256,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
        random_state=RANDOM_STATE,
        verbose=False,
    )),
])
mlp_pipe.fit(X_train, y_train)
val_acc_mlp = mlp_pipe.score(X_val, y_val)
results["MLP"] = {"model": mlp_pipe, "val_acc": val_acc_mlp}
print(f"  Val accuracy: {val_acc_mlp:.4f}\n")

# ── Summary ───────────────────────────────────────────────────────────────────
summary = pd.DataFrame(
    [(name, r["val_acc"]) for name, r in results.items()],
    columns=["Classifier", "Val Accuracy"],
).sort_values("Val Accuracy", ascending=False)
print(summary.to_string(index=False))

## 8 — Save best model

Three files are written to `MODELS_DIR`:

| File | Contents |
|---|---|
| `efficientnet_{b}__{clf}__{ts}.pkl` | The full sklearn Pipeline (scaler + classifier) |
| `label_encoder_sklearn__{ts}.pkl` | The `LabelEncoder` mapping integer indices → species names |
| `config_sklearn__{ts}.json` | Backbone name, image size, feature dim — needed by the API to rebuild the preprocessing step |

The best classifier is picked automatically from the `results` dict.

In [ ]:
import json
from datetime import datetime

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Pick best classifier ──────────────────────────────────────────────────────
best_name  = max(results, key=lambda k: results[k]["val_acc"])
best_model = results[best_name]["model"]
best_acc   = results[best_name]["val_acc"]
print(f"Best classifier : {best_name}  (val_acc={best_acc:.4f})")

# ── Timestamp + slug ─────────────────────────────────────────────────────────
ts   = datetime.now().strftime("%Y%m%d_%H%M")
slug = best_name.lower().replace(" ", "_").replace("(", "").replace(")", "")

pipeline_path = MODELS_DIR / f"efficientnet_{BACKBONE}__{slug}__{ts}.pkl"
encoder_path  = MODELS_DIR / f"label_encoder_sklearn__{ts}.pkl"
config_path   = MODELS_DIR / f"config_sklearn__{ts}.json"

# ── Save pipeline ─────────────────────────────────────────────────────────────
with open(pipeline_path, "wb") as f:
    pickle.dump(best_model, f)

# ── Save label encoder ────────────────────────────────────────────────────────
with open(encoder_path, "wb") as f:
    pickle.dump(le, f)

# ── Save config (for API preprocessing) ──────────────────────────────────────
config = {
    "backbone":    f"efficientnet_{BACKBONE}",
    "img_size":    IMG_SIZE,
    "feat_dim":    FEAT_DIM,
    "classifier":  best_name,
    "val_accuracy": round(best_acc, 4),
    "num_classes": NUM_CLASSES,
    "classes":     list(le.classes_),
    "pipeline_file":  pipeline_path.name,
    "encoder_file":   encoder_path.name,
    "trained_at":  ts,
}
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved → {pipeline_path.name}")
print(f"Saved → {encoder_path.name}")
print(f"Saved → {config_path.name}")